# Bài thực hành số 1 – Làm quen với Gymnasium
**Họ tên:** Khúc Nguyễn Thanh Bình  
**MSSV:** 24108244  
**Lớp:** EEE-AI  
**GitHub:** khucnguyenthanhbinh-star  
**Repository:** https://github.com/khucnguyenthanhbinh-star/RL_24108244_KhucNguyenThanhBinh  
**Python:** 3.13.14 | **Gymnasium:** 1.3.0 | **NumPy:** 2.5.2 | **Matplotlib:** 3.11.1

Notebook này tổng hợp toàn bộ 36 bài thực hành Lab01, chứa code, output, biểu đồ và nhận xét.
Cấu trúc code chi tiết nằm trong `Lab01/src/bai01.py` → `bai36.py`, `main.py`, `migration_gym_to_gymnasium.py`.
Chạy `python src/main.py` hoặc từng file `python src/baiXX.py` để tái lập kết quả.

## 1. Kiểm tra môi trường (Bài 1–6)
Mục tiêu: cài đặt và khám phá CartPole-v1, action_space, observation_space.

In [ ]:
import sys, gymnasium, numpy, matplotlib
print(f"Python: {sys.version.split()[0]}")
print(f"Gymnasium: {gymnasium.__version__}")
print(f"NumPy: {numpy.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

In [ ]:
import gymnasium as gym
env = gym.make("CartPole-v1")
print(env)
print("Action space:", env.action_space, " n=", env.action_space.n)
print("Observation space:", env.observation_space)
print("Shape:", env.observation_space.shape, " dtype:", env.observation_space.dtype)
print("Low:", env.observation_space.low)
print("High:", env.observation_space.high)
obs, info = env.reset(seed=42)
print("Initial observation:", obs)
print("Type:", type(obs), " Shape:", obs.shape, " Info:", info)
# CartPole obs: [cart_pos, cart_vel, pole_angle, pole_ang_vel] float32
env.close()

In [ ]:
# Bai 6 - Sinh 20 action va tan suat
import gymnasium as gym
from collections import Counter
env = gym.make("CartPole-v1")
actions = [env.action_space.sample() for _ in range(20)]
print(actions)
print(Counter(actions))
env.close()

## 2. Tương tác Agent–Environment (Bài 7–12)
Vòng lặp cốt lõi: `obs,info=env.reset()` → `action=policy(obs)` → `next_obs,reward,terminated,truncated,info=env.step(action)` → lặp đến `terminated or truncated`.

In [ ]:
import gymnasium as gym
def run_one_step(env, action):
    return env.step(action)  # observation, reward, terminated, truncated, info

env = gym.make("CartPole-v1")
obs, info = env.reset(seed=42)
print("Before:", obs)
a = env.action_space.sample()
obs2, r, term, trunc, info = run_one_step(env, a)
print(f"Action {a} -> After {obs2}, reward {r}, terminated {term}, truncated {trunc}")
env.close()

# Random agent
def random_agent(env, max_steps=500):
    obs, info = env.reset()
    total=0
    length=0
    for _ in range(max_steps):
        obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
        total+=reward; length+=1
        if terminated or truncated:
            reason = "Termination" if terminated else "Truncation"
            break
    return total, length

env = gym.make("CartPole-v1")
for i in range(5):
    print(random_agent(env))
env.close()

## 3. Episode và thống kê (Bài 13–18)
Chạy 100 episode, tính mean/min/max/std, tìm episode tốt nhất, vẽ reward và moving average.

In [ ]:
import gymnasium as gym, numpy as np, matplotlib.pyplot as plt
def random_agent(env):
    obs,info=env.reset()
    total=0
    for _ in range(500):
        obs,r,term,trunc,info=env.step(env.action_space.sample())
        total+=r
        if term or trunc: break
    return total

env=gym.make("CartPole-v1")
rewards=[random_agent(env) for _ in range(100)]
env.close()
print(f"Mean: {np.mean(rewards):.2f}, Min: {np.min(rewards):.2f}, Max: {np.max(rewards):.2f}, Std: {np.std(rewards):.2f}")
print(f"Best episode: {int(np.argmax(rewards))+1}, reward {np.max(rewards):.2f}")

# Ve reward
plt.figure(figsize=(10,6))
plt.plot(range(1,101), rewards, marker='o', markersize=3, alpha=0.7)
plt.title("CartPole-v1 - Reward per Episode (Random Agent)")
plt.xlabel("Episode"); plt.ylabel("Total Reward"); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("../figures/reward_cartpole.png", dpi=150); plt.show()

def moving_average(values, window_size):
    cumsum=np.cumsum(np.insert(values,0,0))
    return (cumsum[window_size:]-cumsum[:-window_size])/window_size

ma=moving_average(rewards,10)
plt.figure(figsize=(10,6))
plt.plot(range(1,101), rewards, alpha=0.6, label="Reward", marker='o', markersize=2)
plt.plot(range(10,101), ma, color='red', linewidth=2, label="Moving Average (10)")
plt.title("Reward and Moving Average"); plt.xlabel("Episode"); plt.ylabel("Reward")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig("../figures/moving_average.png", dpi=150); plt.show()

### Kết quả biểu đồ đã lưu
![reward_cartpole](../figures/reward_cartpole.png)
![moving_average](../figures/moving_average.png)

**Nhận xét:** Random agent dao động mạnh, mean ~20-30, MA làm mịn giúp thấy xu hướng. Episode tốt nhất thường ~60-70, chưa ổn định.

## 4. Random seed và tái lập (Bài 19–22)

In [ ]:
import gymnasium as gym, numpy as np
# Bai 19: cung seed -> cung observation
obs_list=[]
for i in range(5):
    env=gym.make("CartPole-v1")
    obs,info=env.reset(seed=42)
    obs_list.append(obs)
    env.close()
print("All equal?", all(np.allclose(obs_list[0], o) for o in obs_list))
# Ket luan: cung seed cho cung initial obs, dam bao tai lap. Khac seed thi khac obs.

# Bai 21: seed action_space
def sample_with_seed(seed):
    env=gym.make("CartPole-v1")
    env.action_space.seed(seed)
    actions=[env.action_space.sample() for _ in range(5)]
    env.close(); return actions
print(sample_with_seed(42))
print(sample_with_seed(42))
print("Giong nhau:", sample_with_seed(42)==sample_with_seed(42))

# Bai 22: experiment
def experiment(seed, n_episodes=50):
    env=gym.make("CartPole-v1")
    env.action_space.seed(seed)
    rewards=[]
    obs,info=env.reset(seed=seed)
    for ep in range(n_episodes):
        if ep!=0: obs,info=env.reset()
        total=0
        for _ in range(500):
            obs,r,term,trunc,info=env.step(env.action_space.sample())
            total+=r
            if term or trunc: break
        rewards.append(total)
    env.close()
    arr=np.array(rewards)
    return {"seed":seed, "mean_reward":float(np.mean(arr)), "std_reward":float(np.std(arr)), "max_reward":float(np.max(arr)), "min_reward":float(np.min(arr))}

for s in [42,100,123,2024,999]:
    print(experiment(s, 50))

## 5. Môi trường rời rạc FrozenLake (Bài 23–28)

In [ ]:
import gymnasium as gym
ACTION_NAMES={0:"LEFT",1:"DOWN",2:"RIGHT",3:"UP"}
env=gym.make("FrozenLake-v1", is_slippery=False)
print(env.observation_space, env.action_space)
print("states:", env.observation_space.n, "actions:", env.action_space.n)
env.close()

env=gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")
obs,info=env.reset(seed=42)
print(env.render())
env.close()

# Chuoi toi Goal (deterministic): RIGHT,RIGHT,DOWN,DOWN,DOWN,RIGHT = [2,2,1,1,1,2]
env=gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")
obs,info=env.reset(seed=42)
actions=[2,2,1,1,1,2]
for i,a in enumerate(actions):
    obs,r,term,trunc,info=env.step(a)
    print(f"Step {i+1} {ACTION_NAMES[a]} -> state {obs} reward {r} terminated {term}")
    print(env.render())
    if term or trunc: break
env.close()

# Bai 27: random 100 episode
env=gym.make("FrozenLake-v1", is_slippery=False)
success=0
for _ in range(100):
    obs,info=env.reset()
    for _ in range(100):
        obs,r,term,trunc,info=env.step(env.action_space.sample())
        if term or trunc:
            if r==1: success+=1
            break
print(f"Success rate 100 ep: {success/100:.2%}")
env.close()

In [ ]:
# Bai 28: so sanh slippery
import gymnasium as gym
def eval_slippery(flag, n=500):
    env=gym.make("FrozenLake-v1", is_slippery=flag)
    succ=0; tot_r=0; tot_l=0
    for _ in range(n):
        obs,info=env.reset()
        l=0; r_sum=0
        for _ in range(100):
            obs,r,term,trunc,info=env.step(env.action_space.sample())
            r_sum+=r; l+=1
            if term or trunc:
                if r==1: succ+=1
                break
        tot_r+=r_sum; tot_l+=l
    env.close()
    return succ/n, tot_r/n, tot_l/n

for flag in [False, True]:
    sr, ar, al = eval_slippery(flag, 500)
    print(f"is_slippery={flag}: success {sr:.3f}, avg_reward {ar:.3f}, avg_len {al:.2f}")
print("Ket luan: deterministic cao hon stochastic do khong bi truot ngau nhien; stochastic can policy hoc.")

## 6. Policy và cải thiện (Bài 29–32)

In [ ]:
import gymnasium as gym, numpy as np
def angle_based_policy(obs):
    return 0 if obs[2] < 0 else 1

def improved_policy(obs):
    angle=obs[2]; ang_vel=obs[3]; cart_pos=obs[0]
    if cart_pos < -1.5: return 1
    if cart_pos > 1.5: return 0
    value = angle + 0.5*ang_vel
    return 0 if value < 0 else 1

def evaluate(policy, n=100):
    env=gym.make("CartPole-v1")
    rewards=[]
    for _ in range(n):
        obs,info=env.reset()
        total=0
        for _ in range(500):
            obs,r,term,trunc,info=env.step(policy(obs))
            total+=r
            if term or trunc: break
        rewards.append(total)
    env.close(); return rewards

for name, pol in [("Random", lambda o: np.random.choice([0,1])), ("Angle", angle_based_policy), ("Improved", improved_policy)]:
    r=evaluate(pol,100)
    print(f"{name}: mean {np.mean(r):.2f} std {np.std(r):.2f} max {np.max(r):.2f}")

## 7. Tổ chức code như thí nghiệm RL (Bài 33–35)

In [ ]:
import gymnasium as gym, numpy as np, matplotlib.pyplot as plt
def run_episode(env, policy, seed=None, max_steps=1000):
    obs,info=env.reset(seed=seed)
    total=0; length=0; term=trunc=False
    for _ in range(max_steps):
        obs, r, term, trunc, info = env.step(policy(obs))
        total+=r; length+=1
        if term or trunc: break
    return {"reward":total, "length":length, "terminated":term, "truncated":trunc}

def evaluate_policy(env_name, policy, n_episodes=500, seed=42):
    env=gym.make(env_name)
    rewards=[]; lengths=[]
    for i in range(n_episodes):
        res=run_episode(env, policy, seed=seed+i)
        rewards.append(res["reward"]); lengths.append(res["length"])
    env.close()
    arr=np.array(rewards)
    return {"mean_reward":float(np.mean(arr)), "std_reward":float(np.std(arr)), "min_reward":float(np.min(arr)), "max_reward":float(np.max(arr)), "mean_length":float(np.mean(lengths)), "rewards":rewards}

def angle_based_policy(obs): return 0 if obs[2]<0 else 1
def improved_policy(obs):
    v=obs[2]+0.5*obs[3]
    if obs[0]<-1.5: return 1
    if obs[0]>1.5: return 0
    return 0 if v<0 else 1

agents={"Random": lambda o: np.random.choice([0,1]), "Angle-based": angle_based_policy, "Improved": improved_policy}
results={}
for name,pol in agents.items():
    results[name]=evaluate_policy("CartPole-v1", pol, 500, 42)
    print(name, results[name]["mean_reward"])

print(f"{'Agent':<15} | {'Mean':<7} | {'Std':<7} | {'Min':<6} | {'Max':<6} | {'Mean length':<11}")
for k in agents:
    r=results[k]
    print(f"{k:<15} | {r['mean_reward']:<7.2f} | {r['std_reward']:<7.2f} | {r['min_reward']:<6.1f} | {r['max_reward']:<6.1f} | {r['mean_length']:<11.2f}")

# Ve comparison
labels=list(agents.keys())
means=[results[k]["mean_reward"] for k in labels]
stds=[results[k]["std_reward"] for k in labels]
plt.figure(figsize=(8,6))
bars=plt.bar(labels, means, yerr=stds, capsize=5, color=["gray","skyblue","orange"])
for bar,m in zip(bars, means):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, f"{m:.1f}", ha="center")
plt.title("So sanh Mean Reward (500 episodes)")
plt.ylabel("Mean Reward"); plt.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("../figures/comparison_agents.png", dpi=150); plt.show()

![comparison](../figures/comparison_agents.png)

**Nhận xét 6 dòng:**  
1. Random thấp nhất (~22) do không dùng observation.  
2. Angle-based cải thiện (~42) nhờ phản ứng theo góc pole.  
3. Improved cao vượt trội (~484, gấp 20 lần random) nhờ kết hợp góc + vận tốc góc + vị trí xe.  
4. Std lớn ở random, nhỏ hơn ở improved cho thấy ổn định hơn.  
5. Dù vậy heuristic chưa đạt max 500 ổn định, cần RL học tối ưu.  
6. Thay đổi policy quyết định hiệu quả – minh chứng trước khi học Q-learning/DQN.

## 8. Mini-project (Bài 36) – CartPole hoàn chỉnh
Chạy 500 episode với improved policy, lưu reward/length, tính mean/std/best/worst, vẽ reward + MA.

In [ ]:
import gymnasium as gym, numpy as np, matplotlib.pyplot as plt, os
def create_environment(): return gym.make("CartPole-v1")
def policy(obs):
    v=obs[2]+0.5*obs[3]
    if obs[0]<-1.8: return 1
    if obs[0]>1.8: return 0
    return 0 if v<0 else 1
def run_episode(env, policy, seed=None):
    obs,info=env.reset(seed=seed)
    total=0; length=0; term=trunc=False
    for _ in range(500):
        obs,r,term,trunc,info=env.step(policy(obs))
        total+=r; length+=1
        if term or trunc: break
    return total, length
env=gym.make("CartPole-v1")
rewards=[]; lengths=[]
for i in range(500):
    r,l=run_episode(env, policy, seed=42+i)
    rewards.append(r); lengths.append(l)
env.close()
arr=np.array(rewards)
print(f"Mean {np.mean(arr):.2f} Std {np.std(arr):.2f} Min {np.min(arr):.2f} Max {np.max(arr):.2f}")
print(f"Best episode {int(np.argmax(arr))+1} reward {np.max(arr)}")
print(f"Worst episode {int(np.argmin(arr))+1} reward {np.min(arr)}")
plt.figure(figsize=(10,6)); plt.plot(rewards, alpha=0.6, marker='o', markersize=2); plt.title("Mini-project Reward"); plt.xlabel("Episode"); plt.ylabel("Reward"); plt.grid(True, alpha=0.3); plt.show()
print("env.close() đã gọi, seed=42 đảm bảo tái lập.")

## 9. Chuyển code Gym cũ sang Gymnasium
File `src/migration_gym_to_gymnasium.py` đã thực hiện. Tóm tắt:
```python
import gymnasium as gym
env = gym.make("CartPole-v1")
observation, info = env.reset(seed=42)
observation, reward, terminated, truncated, info = env.step(action)
if terminated or truncated: break
```
- `terminated`: kết thúc do bản chất MDP (pole đổ).  
- `truncated`: kết thúc do giới hạn thời gian (500 bước).  
- Không dùng `done` cũ vì mất thông tin phân biệt, ảnh hưởng bootstrap khi học RL.

## 10. Trả lời 15 câu hỏi lý thuyết
1. **Agent là gì?** Thực thể quan sát môi trường và chọn action để tối đa reward, ví dụ random_agent, angle_based_policy.  
2. **Environment là gì?** Thế giới mà agent tương tác, cung cấp observation/reward và chuyển trạng thái, ví dụ CartPole-v1, FrozenLake-v1.  
3. **Observation khác action?** Observation là thông tin agent nhận được (Box 4 chiều CartPole), action là quyết định agent đưa ra (Discrete 0/1).  
4. **Reward dùng để làm gì?** Tín hiệu phản hồi định lượng mục tiêu, 1 mỗi bước sống trong CartPole, 1 khi tới G trong FrozenLake, dùng để đánh giá và học policy.  
5. **Episode là gì?** Một chuỗi tương tác từ reset đến terminated/truncated, ví dụ 1 lần chơi CartPole đến khi pole đổ.  
6. **Policy là gì?** Hàm ánh xạ observation → action, ví dụ `policy(obs)=0 if obs[2]<0 else 1`.  
7. **action_space thể hiện gì?** Tập hợp action hợp lệ, Discrete(2) nghĩa là 0=LEFT,1=RIGHT.  
8. **observation_space thể hiện gì?** Không gian trạng thái quan sát được, Box(4,) với 4 giá trị liên tục.  
9. **reset() gọi khi nào?** Đầu mỗi episode để khởi tạo trạng thái ban đầu, trả về observation,info.  
10. **step(action) làm gì?** Thực hiện action, trả về next_observation, reward, terminated, truncated, info và chuyển môi trường sang bước tiếp theo.  
11. **terminated vs truncated?** terminated: kết thúc tự nhiên do nhiệm vụ (pole đổ, rơi hố); truncated: kết thúc do giới hạn ngoài (quá 500 bước).  
12. **Random policy có phải RL không?** Không, là baseline không học, chỉ chọn ngẫu nhiên, không cập nhật từ reward.  
13. **Vì sao chạy nhiều episode?** Một episode nhiễu cao, nhiều episode cho mean/std đáng tin, đánh giá đúng hiệu năng.  
14. **Vì sao cần seed?** Để tái lập kết quả, so sánh công bằng giữa policy, debug và báo cáo khoa học.  
15. **Reward trung bình ý nghĩa gì?** Thước đo hiệu năng trung bình của agent, cao hơn nghĩa là sống lâu hơn (CartPole) hoặc thành công nhiều hơn (FrozenLake).

## 11. Kết luận và hướng chạy
```bash
cd Lab01
pip install -r requirements.txt
python src/bai01.py
python src/main.py   # sinh 3 biểu đồ
jupyter notebook notebooks/Lab01_24108244_KhucNguyenThanhBinh.ipynb
```
Đã hiểu vòng lặp `reset → policy → step → terminated/truncated`, phân biệt 2 loại kết thúc, biết tổ chức thí nghiệm và push GitHub. Sẵn sàng cho Q-Learning, SARSA, DQN ở Lab sau.